# Day 31 — Practice Questions
### What Machine Learning Is · Module 3 · Topics 3.1 + 3.2

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything.

This is your first ML practice set and **no prior knowledge is assumed**. Three questions
to keep asking yourself:

- Is this **supervised or unsupervised**? If supervised, **regression or classification**?
- Is this column **nominal or ordinal**?
- Did I **split before I encoded**?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.simplefilter("ignore")

print("ready")

---
# Part A — Predict the output

Eight short cells. Write your prediction first, then run it.

### A1 · What shape comes out of one-hot encoding?

In [ ]:
d = pd.DataFrame({"colour": ["red", "blue", "red", "green"],
                  "size":   ["S", "M", "L", "S"],
                  "n":      [1, 2, 3, 4]})

print(d.shape)
print(pd.get_dummies(d, columns=["colour", "size"]).shape)
print(pd.get_dummies(d, columns=["colour", "size"], drop_first=True).shape)

### A2 · Which rows get a 1?

In [ ]:
small = pd.DataFrame({"city": ["Boston", "Austin", "Miami", "Austin"]})
out = pd.get_dummies(small, columns=["city"])
print(out.astype(int))
print("\nsum of each row:", out.astype(int).sum(axis=1).tolist())

### A3 · What order does LabelEncoder choose?

In [ ]:
le = LabelEncoder().fit(["medium", "low", "high"])
print(dict(zip(le.classes_, range(3))))

### A4 · Train/test split shapes

In [ ]:
X = np.arange(200).reshape(-1, 2)      # 100 rows, 2 columns
y = np.arange(100)

for ts in [0.2, 0.25, 0.33]:
    X_tr, X_te, _, _ = train_test_split(X, y, test_size=ts, random_state=0)
    print(f"test_size={ts}  ->  train {X_tr.shape}, test {X_te.shape}")

### A5 · What happens with a lot of categories?

In [ ]:
rng = np.random.default_rng(7)
N = 1000
zipcode = rng.integers(10000, 10800, N).astype(str)

print("rows:              ", N)
print("unique zip codes:  ", len(set(zipcode)))
print("columns after one-hot:", pd.get_dummies(pd.DataFrame({"z": zipcode}),
                                               columns=["z"]).shape[1])

### A6 · A column with only one value

In [ ]:
d = pd.DataFrame({"country": ["US"]*5, "sales": [10, 20, 30, 40, 50]})
out = pd.get_dummies(d, columns=["country"])
print(out.astype({"country_US": int}))
print("\ncolumns:", list(out.columns))

### A7 · Train score vs test score as a tree gets deeper

In [ ]:
rng = np.random.default_rng(5101)
n = 40
x = rng.uniform(0, 10, n)
y = 2*x + rng.normal(0, 4, n)
X_tr, X_te, y_tr, y_te = train_test_split(x.reshape(-1, 1), y, test_size=0.3, random_state=1)

print(f"{'max_depth':>10} | {'train R2':>9} | {'test R2':>8}")
print("-"*33)
for depth in [1, 2, 3, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_tr, y_tr)
    print(f"{str(depth):>10} | {m.score(X_tr, y_tr):>9.3f} | {m.score(X_te, y_te):>8.3f}")

### A8 · Classify these five problems

In [ ]:
problems = [
    "Predict tomorrow's electricity demand in megawatts",
    "Decide whether a transaction is fraudulent",
    "Group 50,000 shoppers by their buying behaviour, no labels given",
    "Predict how many days until a machine fails",
    "Teach a drone to land, rewarding soft landings",
]
for p in problems:
    print("-", p)

**For each one: supervised or unsupervised (or reinforcement)? If supervised, regression or classification?**

---
# Part B — Spot & fix the bug

Each cell runs without error and produces a confident, wrong result.

### B1 · "The brand doesn't seem to matter much"

An analyst label-encodes the brand column and reports a weak model.

In [ ]:
rng = np.random.default_rng(5110)
n = 500
brands = ["Acme", "Globex", "Initech", "Umbrella", "Soylent"]
TRUE_EFFECT = {"Acme": 10, "Globex": 55, "Initech": 30, "Umbrella": 5, "Soylent": 42}

brand = rng.choice(brands, n)
age   = rng.uniform(0, 10, n)
value = np.array([100 - 6*a + TRUE_EFFECT[b] for a, b in zip(age, brand)]) + rng.normal(0, 8, n)
df = pd.DataFrame({"brand": brand, "age": age, "value": value})

df["brand_le"] = LabelEncoder().fit_transform(df["brand"])
tr, te = train_test_split(df.index, test_size=0.3, random_state=2)

m = LinearRegression().fit(df.loc[tr, ["age", "brand_le"]], df.loc[tr, "value"])
pred = m.predict(df.loc[te, ["age", "brand_le"]])
print(f"test MAE {mean_absolute_error(df.loc[te, 'value'], pred):.2f}, "
      f"R2 {r2_score(df.loc[te, 'value'], pred):.3f}")
print("Conclusion: brand is only weakly related to value.")

**What is wrong, and what is the real answer?**

### B2 · "Our model is 100% accurate"

In [ ]:
rng = np.random.default_rng(5111)
n = 60
x = rng.uniform(0, 10, n)
y = 3*x + rng.normal(0, 5, n)

model = DecisionTreeRegressor(random_state=0)
model.fit(x.reshape(-1, 1), y)
print(f"R2 = {model.score(x.reshape(-1, 1), y):.3f}")
print("Our model explains 100% of the variance. Ship it.")

**What is wrong, and what is the model's real performance?**

### B3 · "Target encoding gave us a big improvement"

In [ ]:
rng = np.random.default_rng(5112)
N, k = 500, 120
d = pd.DataFrame({"cat": rng.integers(0, k, N).astype(str),
                  "y":   rng.normal(0, 1, N)})      # NOTE: no relationship at all

# encode the category by its average target, using the whole dataset
means = d.groupby("cat").y.mean()
d["cat_encoded"] = d.cat.map(means)

tr, te = train_test_split(d, test_size=0.3, random_state=4)
m = LinearRegression().fit(tr[["cat_encoded"]], tr.y)
print(f"test R2 = {m.score(te[['cat_encoded']], te.y):+.3f}")
print("Target encoding found real signal in the category column.")

**What is wrong? (Hint: look at how `d` was built.)**

### B4 · "It worked in training and crashed in production"

In [ ]:
train = pd.DataFrame({"city": ["A", "B", "A", "B"], "v": [1, 2, 3, 4]})
test  = pd.DataFrame({"city": ["A", "C"],           "v": [5, 6]})     # 'C' is new!

X_tr = pd.get_dummies(train, columns=["city"]).astype(float)
X_te = pd.get_dummies(test,  columns=["city"]).astype(float)
print("train columns:", list(X_tr.columns))
print("test  columns:", list(X_te.columns))

m = LinearRegression().fit(X_tr, [10, 20, 30, 40])
try:
    print(m.predict(X_te))
except ValueError as e:
    print("\nValueError:", str(e)[:80])

**Why did it crash, and how do you fix it?**

---
# Part C — Write the code

Five tasks. Write the code, then run the check underneath.

### C1 · Classify every column

Write `column_report(df)` that returns, for each column: its dtype, how many distinct values
it has, and your suggested treatment — `numeric`, `nominal`, or `high-cardinality`
(more than 15 distinct values and not numeric).

In [ ]:
rng = np.random.default_rng(5120)
N = 800
raw = pd.DataFrame({
    "area":    rng.uniform(500, 2500, N).round(0),
    "city":    rng.choice(["Boston", "Austin", "Denver", "Miami"], N, p=[.3, .3, .25, .15]),
    "size":    rng.choice(["S", "M", "L", "XL"], N),
    "zipcode": rng.integers(1000, 1400, N).astype(str),
    "price":   rng.uniform(100, 400, N).round(1),
})

def column_report(df):
    # your code here
    pass

### C2 · Group rare categories

Write `group_rare(series, max_cats)` that keeps the `max_cats` most common values and
replaces everything else with `"Other"`. Run it on `zipcode`.

In [ ]:
def group_rare(series, max_cats=15):
    # your code here
    pass

### C3 · An ordinal map you write yourself

Write `encode_ordinal(df, col, order)` that maps a column onto 0, 1, 2, … following the
order **you** supply, and raises a clear error if the data contains a value not in your list.

In [ ]:
def encode_ordinal(df, col, order):
    # your code here
    pass

### C4 · The full `prepare()` function

Write `prepare(df, label, ordinal_maps, max_cats)` that does all four steps in the right
order: **split**, ordinal-map, group rare, one-hot + align. Then run it on `raw` with a
realistic price column and report the shapes.

In [ ]:
# a version of `raw` with a price that actually depends on the features
rng = np.random.default_rng(5120)
N = 800
city = rng.choice(["Boston", "Austin", "Denver", "Miami"], N, p=[.3, .3, .25, .15])
size = rng.choice(["S", "M", "L", "XL"], N)
zipc = rng.integers(1000, 1400, N).astype(str)
area = rng.uniform(500, 2500, N)
SZ = {"S": 0, "M": 1, "L": 2, "XL": 3}
CT = {"Boston": 120, "Austin": 40, "Denver": 65, "Miami": 95}
price = (30 + 0.09*area + np.array([CT[c] for c in city])
         + np.array([12*SZ[s] for s in size]) + rng.normal(0, 14, N))
raw = pd.DataFrame({"area": area.round(0), "city": city, "size": size,
                    "zipcode": zipc, "price": price.round(1)})

def prepare(df, label, ordinal_maps=None, max_cats=15, test_size=0.25, random_state=42):
    # your code here
    pass

### C5 · Compare three encodings of the same column

`size` is ordinal (S < M < L < XL). Encode it three ways — your ordinal map, one-hot, and
(wrongly) `LabelEncoder` — and compare the test error of each.

In [ ]:
# your code here

---
# Part D — Challenge

### D1 · Classify eight real problems

For each, state: **(a)** supervised / unsupervised / reinforcement, **(b)** if supervised,
regression or classification, **(c)** what one row of the training data would look like, and
**(d)** one thing that could go wrong.

1. A bank wants to predict which loan applicants will default.
2. A hospital wants to predict a patient's length of stay in days.
3. A retailer has 2 million transactions and wants to find products bought together.
4. A factory wants to predict which of 6 fault types a vibration pattern indicates.
5. A news site wants to order its homepage so people read more articles.
6. A company has 120 survey questions and wants to describe employees in 4 scores.
7. A utility wants to predict tomorrow's peak electricity demand in megawatts.
8. A hospital wants to flag patients likely to be readmitted within 30 days.

In [ ]:
# Use this cell for any checking you want to do

### D2 · Fix a broken pipeline

Here is a data preparation function written by somebody on their first week. It runs. It
produces a model with a good-looking score. **It has four separate bugs.**

Find all four, say what each one does, and write a corrected version.

In [ ]:
def prepare_broken(df, label):
    y = df[label]
    X = df.drop(columns=[label])

    # encode all the categorical columns
    for col in X.select_dtypes(include="object").columns:
        X[col] = LabelEncoder().fit_transform(X[col])

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25)

    return X_tr, X_te, y_tr, y_te


Xb_tr, Xb_te, yb_tr, yb_te = prepare_broken(raw.copy(), "price")
mb = LinearRegression().fit(Xb_tr, yb_tr)
print(f"train R2 {mb.score(Xb_tr, yb_tr):.3f}   test R2 {mb.score(Xb_te, yb_te):.3f}")

In [ ]:
# your corrected version here

---
### Before you hand in

- For every dataset, did you **split before encoding anything**?
- Did you check whether each categorical column is **nominal or ordinal** — and write the
  ordinal order yourself?
- Did you report the **test** score, never the training score?
- Did you look at the **gap** between train and test, not just the test number?
- Did you set a **`random_state`** so your results reproduce?
- For Part D, did you find all four bugs? Both D tasks are marked.

Solutions will be released after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*